Цель работы: Используя сырые данные генотипирования 23andMe, определить гено- и фенотипические характиристики индивида, а также предложить

Методы исседования и их назначение:


1. PLINK 1.9 – конвертация 23andMe в VCF
2. bcftools – фильтрация VCF
3. snpEff / SnpSift – аннотация вариантов, поиск по ClinVar и GWAS Catalog
4. Ensembl VEP (веб) – дополнительная аннотация
5. JamesLick mthap – определение митохондриальной гаплогруппы
6. MorleyDNA Y‑Tree – определение Y‑гаплогруппы
7. ClinVar (ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh37/)
8. GWAS Catalog (www.ebi.ac.uk/gwas/api/search/downloads/full)



Шаг №1: подготовка данных

In [ ]:
head -n 5 genome_23andme.txt # смотрим формат и первые строки

SyntaxError: invalid syntax (4209184052.py, line 1)

Шаг №2: Конвертация в VCF и фильтрация

In [ ]:
plink --23file genome_23andme.txt --recode vcf --out snps_raw --output-chr MT --snps-only just-acgt

In [ ]:
# --23file – входной файл 23andMe.
#--recode vcf – выходной формат VCF.
# --output-chr MT – митохондриальная хромосома обозначается как MT.
#--snps-only just-acgt – исключить инделы, оставить только SNP с аллелями A,C,G,T.

 Шаг №3: Удаление моноаллельных позиций (гомозиготных по референсу)

In [ ]:
bcftools view -i 'GT!="0/0"' snps_raw.vcf > snps_variants.vcf

In [ ]:
# -i 'GT!="0/0"' оставляет записи, где генотип не 0/0 (то есть есть хотя бы один альтернативный аллель).

Шаг №4: статистика количества SNP

In [ ]:
grep -v "^#" snps_variants.vcf | wc -l

Шаг №5: определяем пол (гендер-пати)

In [ ]:
grep -v "^#" snps_variants.vcf | cut -f1 | grep -c "Y"

In [ ]:
# если результат > 0, то индивид – мужчина, для женского пола Y-хромосомные SNP отсутствуют.

Шаг №6: определяем гаплогруппы

6.1. Митохондриальная гаплогруппа (материнская линия)
онлайн сервис James Lick mthap

загружаем исходный файл 23andMe и получаем результат

6.2. Y-хромосомная гаплогруппа (отцовская линия)
сервис MorleyDNA Y‑SNP subclade predictor
также загружаем исподный файл 23andMe

Шаг №7: пркедстаказать цвета глаз

по данным из литературы знаем, что ключевые SNP:
rs12913832 (HERC2) – основной детерминант карих/голубых глаз.

rs12896399 (SLC24A4), rs12203592 (IRF4), rs16891982 (SLC45A2) – уточняют оттенок

In [ ]:
grep -E "rs12913832|rs12896399|rs12203592|rs16891982" genome_23andme.txt # в исходным файле

AG или GG → карие глаза, AA → голубые/зеленые

Шаг №8: аннотация клинически значимых SNP

In [ ]:
# скачиваем snpEff
wget https://snpeff.blob.core.windows.net/versions/snpEff_latest_core.zip
unzip snpEff_latest_core.zip
cd snpEff
# скачиваем базу GRCh37.75
java -jar snpEff.jar download GRCh37.75

In [ ]:
#аннотация SNP с помощью snpEff
java -jar snpEff/snpEff.jar GRCh37.75 snps_variants.vcf > snps_annotated.vcf #

In [ ]:
# скачивание ClinVar VCF для GRCh37
wget https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh37/clinvar.vcf.gz
gunzip clinvar.vcf.gz

In [ ]:
# аннотация с помощью SnpSift
java -jar snpEff/SnpSift.jar annotate clinvar.vcf snps_annotated.vcf > snps_clinvar.vcf

In [ ]:
# извлечение SNP с клинической значимостью
grep -E "CLIN_SIG=.*risk_factor|CLIN_SIG=.*pathogenic" snps_clinvar.vcf > clinically_relevant_snps.vcf

In [ ]:
# или ищем конкретные диагнозы
grep "CLNDN" snps_clinvar.vcf | less

Аннотация через GWAS Catalog

In [ ]:
# скачиваем GWAS Catalog в формате tsv
wget https://www.ebi.ac.uk/gwas/api/search/downloads/full -O gwas_catalog.tsv

In [ ]:
# затем используем SnpSift для добавления GWAS-аннотаций
java -jar snpEff/SnpSift.jar gwasCat -db gwas_catalog.tsv snps_variants.vcf > snps_gwas.vcf

In [ ]:
# извлекаем признаки, ассоциированные с вариантами
grep -v "^#" snps_gwas.vcf | cut -f8 | grep -o "GWASCAT_TRAIT=[^;]*" | sort | uniq

Заканчиваем работу: выбираем аналитически 5 SNP для гипотетического редактирования и мищени для CRISPR